In [1]:
!pip install faiss-cpu pykeops

In [2]:
import torch
import numpy as np
import time
from torchvision import datasets
from pykeops.torch import LazyTensor
import faiss

def compare_pykeops_faiss_mnist(k=5, num_queries=100):
    # -------------------------------
    # 1. Load and preprocess MNIST data.
    # -------------------------------
    train_dataset = datasets.MNIST(root='./data', train=True, download=True)
    test_dataset  = datasets.MNIST(root='./data', train=False, download=True)

    # Flatten images and normalize pixel values.
    train_data = train_dataset.data.float().view(-1, 28*28) / 255.0
    test_data  = test_dataset.data.float().view(-1, 28*28) / 255.0
    X = torch.cat([train_data, test_data], dim=0)  # shape: (70000, 784)
    N, D = X.shape
    print(f"MNIST loaded: {N} points, dimension {D}")

    # -------------------------------
    # 2. Compute exact k-NN using PyKeOps (brute-force).
    # -------------------------------
    # Create LazyTensors for efficient pairwise computation.
    X_i = LazyTensor(X.unsqueeze(1))  # shape: (N, 1, D)
    X_j = LazyTensor(X.unsqueeze(0))  # shape: (1, N, D)
    # Compute squared Euclidean distances.
    D_ij = ((X_i - X_j) ** 2).sum(-1)   # shape: (N, N)

    # Retrieve the k+1 nearest neighbors for each point (first neighbor is the point itself).
    neighbors_pykeops = D_ij.argKmin(k + 1, dim=1)  # shape: (N, k+1)
    # Remove self-match and bring to CPU.
    neighbors_pykeops = neighbors_pykeops[:, 1:].cpu().long()  # shape: (N, k)

    # Compute the corresponding distances using standard PyTorch operations.
    X_neighbors = X[neighbors_pykeops]         # shape: (N, k, D)
    diffs = X.unsqueeze(1) - X_neighbors         # shape: (N, k, D)
    distances_pykeops = (diffs ** 2).sum(-1)     # shape: (N, k)

    # -------------------------------
    # 3. Build FAISS index on MNIST.
    # -------------------------------
    #X_np = X.numpy().astype(np.float32)
    #d = X_np.shape[1]
    #index_faiss = faiss.IndexFlatL2(d)  # L2 distance index
    #index_faiss.add(X_np)

    X_np = X.numpy().astype(np.float32)
    d = X_np.shape[1]
    nlist = 100  # Number of clusters
    quantizer = faiss.IndexFlatL2(d)
    index_faiss = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)

    # Train the index (IVF indices require training).
    index_faiss.train(X_np)
    index_faiss.add(X_np)
    # Set nprobe for a better recall (number of clusters to search).
    index_faiss.nprobe = 10

    # -------------------------------
    # 4. For a subset of queries, compare FAISS and PyKeOps results.
    # -------------------------------
    rng = np.random.default_rng(42)
    query_indices = rng.choice(N, size=num_queries, replace=False)

    overlap_ratios = []
    distance_error_ratios = []
    faiss_runtime = []

    for count, qi in enumerate(query_indices, start=1):
        # Get query vector for FAISS (as NumPy array).
        q_np = X_np[qi]

        # PyKeOps results for this query.
        pykeops_indices = neighbors_pykeops[qi].numpy()  # shape: (k,)
        pykeops_dists   = distances_pykeops[qi].numpy()    # shape: (k,)

        # FAISS exact k-NN search (retrieve k+1 neighbors; remove self-match).
        start_faiss = time.perf_counter_ns()
        D_faiss, I_faiss = index_faiss.search(q_np.reshape(1, -1), k + 1)
        end_faiss = time.perf_counter_ns()
        faiss_runtime.append(end_faiss - start_faiss)

        faiss_indices = I_faiss[0][1:]
        faiss_dists   = D_faiss[0][1:]

        print(f"\nQuery {count}/{num_queries} (Index: {qi}):")
        print("  PyKeOps -> distances:", pykeops_dists, ", indices:", pykeops_indices)
        print("  FAISS   -> distances:", faiss_dists, ", indices:", faiss_indices)

        # Compute the index overlap ratio (fraction of common indices).
        common = set(pykeops_indices).intersection(set(faiss_indices))
        overlap = len(common) / k
        overlap_ratios.append(overlap)
        print(f"  Index overlap ratio: {overlap:.2f}")

        # For distance comparison, sort both distance arrays (order may differ).
        sorted_pykeops = np.sort(pykeops_dists)
        sorted_faiss = np.sort(faiss_dists)
        # Compute element-wise ratio (FAISS / PyKeOps) with care to avoid division by zero.
        ratio = np.divide(sorted_faiss, sorted_pykeops, out=np.ones_like(sorted_faiss), where=(sorted_pykeops != 0))
        mean_ratio = np.mean(ratio)
        distance_error_ratios.append(mean_ratio)
        print(f"  Mean distance error ratio (FAISS/PyKeOps): {mean_ratio:.6f}")

    avg_overlap = np.mean(overlap_ratios)
    avg_distance_ratio = np.mean(distance_error_ratios)
    avg_faiss_time = np.mean(faiss_runtime)

    print("\n=== Summary ===")
    print(f"Average index overlap ratio: {avg_overlap:.6f}")
    print(f"Average distance error ratio (FAISS/PyKeOps): {avg_distance_ratio:.6f}")
    print(f"Average FAISS query runtime: {avg_faiss_time/num_queries:.3f} ns")

    tolerance = 1.10  # Allow up to 10% increase in distance.
    if avg_distance_ratio > tolerance:
        print(f"FAISS's average distance ratio {avg_distance_ratio:.6f} exceeds the tolerance {tolerance}")
    else:
        print(f"FAISS's average distance ratio {avg_distance_ratio:.6f} is within the tolerance {tolerance}")

if __name__ == "__main__":
    compare_pykeops_faiss_mnist(k=50, num_queries=1000)


Streaming output truncated to the last 5000 lines.
 44367 59746]
  FAISS   -> distances: [14.064852 14.884998 15.258686 16.722    16.766321 18.039078 19.037325
 19.108051 19.147423 19.711649 20.050797 20.159016 20.568027 20.714972
 21.150682 21.351206 21.66316  21.92786  22.227606 22.361923 22.429588
 22.611782 22.673435 22.7574   22.766798 22.83236  23.136703 23.302254
 23.335194 23.41298  23.463268 23.628681 23.63934  23.69249  23.797049
 24.004414 24.0219   24.035973 24.354727 24.561787 24.73627  24.77687
 24.828854 24.831343 24.855427 24.87148  24.927614 24.953758 25.043585
 25.065975] , indices: [41461 41549 62123 54937  8287 22279 15991  7783  7769 56077  9927 54971
 57751 67238 44417   583 54917 14597 44117 23911 32247 41615 12373 66609
 41551 13755 41677 41131 30181  3507 19199    79 41379  1057 54995 30022
 34959 22613  7287 43307 24263 17359 31100 44323 44361 53951 29867 31439
 44367 59746]
  Index overlap ratio: 1.00
  Mean distance error ratio (FAISS/PyKeOps): 1.000000

Que

In [2]:
import numpy as np
from numba import njit, prange
import time

# ------------------------------------------------------------------------------
# Numba helper: extract candidate indices for one projection.
# ------------------------------------------------------------------------------
@njit(parallel=True)
def query_candidates_nb(q, sorted_projs, sorted_idxs, proj_dirs, window_size, n, L):
    """
    For a given query q (1D array, float32) and precomputed arrays,
    extract a window of candidate indices from each projection direction.

    Parameters:
      q : (d,) query point (float32)
      sorted_projs : (L, n) sorted projection values per direction.
      sorted_idxs  : (L, n) corresponding sorted indices.
      proj_dirs    : (L, d) projection directions.
      window_size  : half-window size (integer)
      n            : total number of points (integer)
      L            : number of projection directions (integer)

    Returns:
      candidate_indices : 1D array of unique candidate indices.
    """
    # Pre-allocate candidate matrix (L x (2*window_size))
    candidate_matrix = np.empty((L, 2 * window_size), dtype=np.int32)

    # Loop over each projection direction in parallel.
    for i in prange(L):
        # Compute dot product of q and the i-th projection direction.
        dot = 0.0
        for j in range(q.shape[0]):
            dot += proj_dirs[i, j] * q[j]
        q_val = dot
        # Binary search in the sorted projection array.
        idx = np.searchsorted(sorted_projs[i], q_val)
        slice_size = 2 * window_size
        start = idx - window_size
        if start < 0:
            start = 0
        if start > n - slice_size:
            start = n - slice_size
        # Extract candidate indices for this projection.
        candidate_matrix[i] = sorted_idxs[i, start:start + slice_size]

    # Flatten candidate_matrix and compute unique indices.
    flat = candidate_matrix.reshape(L * 2 * window_size)
    flat_sorted = np.sort(flat)
    # Count unique values.
    count = 1
    for i in range(1, flat_sorted.shape[0]):
        if flat_sorted[i] != flat_sorted[i - 1]:
            count += 1
    unique_arr = np.empty(count, dtype=flat_sorted.dtype)
    unique_arr[0] = flat_sorted[0]
    pos = 1
    for i in range(1, flat_sorted.shape[0]):
        if flat_sorted[i] != flat_sorted[i - 1]:
            unique_arr[pos] = flat_sorted[i]
            pos += 1
    return unique_arr

# ------------------------------------------------------------------------------
# Numba helper: compute squared Euclidean distances for candidate indices.
# ------------------------------------------------------------------------------
@njit
def compute_dists(q, data, candidate_indices):
    M = candidate_indices.shape[0]
    d = q.shape[0]
    dists = np.empty(M, dtype=np.float32)
    for i in range(M):
        s = 0.0
        for j in range(d):
            diff = data[candidate_indices[i], j] - q[j]
            s += diff * diff
        dists[i] = s
    return dists

# ------------------------------------------------------------------------------
# AMPIIndex: Adaptive Multi-Projection Index using NumPy and Numba.
# ------------------------------------------------------------------------------
class AMPIIndex:
    def __init__(self, data, num_projections):
        """
        Initializes the AMPI index.

        Args:
          data: A NumPy array of shape (n, d) containing n data points.
          num_projections: The number L of random projection directions.
        """
        self.data = data  # shape (n, d)
        self.n, self.d = data.shape
        self.L = num_projections

        rng = np.random.default_rng(0)
        # Generate L random projection directions (each normalized)
        raw_dirs = rng.standard_normal((self.L, self.d)).astype(np.float32)
        norms = np.linalg.norm(raw_dirs, axis=1, keepdims=True)
        self.proj_dirs = raw_dirs / norms  # shape (L, d)

        # Precompute projections for each direction.
        self.sorted_projs = np.empty((self.L, self.n), dtype=np.float32)
        self.sorted_idxs = np.empty((self.L, self.n), dtype=np.int32)
        for i in range(self.L):
            proj = data @ self.proj_dirs[i]  # (n,)
            sorted_idx = np.argsort(proj)
            self.sorted_projs[i] = proj[sorted_idx]
            self.sorted_idxs[i] = sorted_idx

    def query_candidates(self, q, window_size=10):
        """
        For query q, extract candidate indices using the projection-based search.

        Args:
          q: Query point of shape (d,) (NumPy array, float32)
          window_size: half-window size.

        Returns:
          candidate_indices: 1D NumPy array of unique candidate indices.
        """
        # Ensure q is a float32 1D array.
        q = np.asarray(q, dtype=np.float32)
        candidate_indices = query_candidates_nb(q, self.sorted_projs, self.sorted_idxs,
                                                self.proj_dirs, window_size, self.n, self.L)
        return candidate_indices

    def query(self, q, window_size=10, k=1):
        """
        Computes the k-NN result:
          1. Extract candidate indices.
          2. Compute full L2 distances on candidate set.
          3. Return top k neighbors.

        Returns:
          final_points: (k, d) array of k nearest neighbors.
          final_dists: (k,) array of squared L2 distances.
          candidate_indices: candidate set used.
        """
        candidate_indices = self.query_candidates(q, window_size=window_size)
        dists = compute_dists(q, self.data, candidate_indices)
        sorted_order = np.argsort(dists)
        final_indices = candidate_indices[sorted_order][:k]
        final_points = self.data[final_indices]
        final_dists = dists[sorted_order][:k]
        return final_points, final_dists, final_indices

In [3]:
import torch
import numpy as np
import time
from torchvision import datasets
from pykeops.torch import LazyTensor

def compare_ampi_pykeops_mnist(k=5, num_projections=100, window_size=100, num_queries=100):
    # -------------------------------
    # 1. Load and preprocess MNIST data.
    # -------------------------------
    train_dataset = datasets.MNIST(root='./data', train=True, download=True)
    test_dataset  = datasets.MNIST(root='./data', train=False, download=True)

    # Flatten images and normalize pixel values.
    train_data = train_dataset.data.float().view(-1, 28*28) / 255.0
    test_data  = test_dataset.data.float().view(-1, 28*28) / 255.0
    X = torch.cat([train_data, test_data], dim=0)  # shape: (70000, 784)
    N, D = X.shape
    print(f"MNIST loaded: {N} points, dimension {D}")

    # -------------------------------
    # 2. Compute exact k-NN using PyKeOps (brute-force).
    # -------------------------------
    # Create LazyTensors for efficient (memory-friendly) pairwise computation.
    X_i = LazyTensor(X.unsqueeze(1))  # shape: (N, 1, D)
    X_j = LazyTensor(X.unsqueeze(0))  # shape: (1, N, D)
    # Compute squared Euclidean distances.
    D_ij = ((X_i - X_j) ** 2).sum(-1)   # shape: (N, N)

    # Retrieve the k+1 nearest neighbors for each point (first neighbor is the point itself).
    neighbors_pykeops = D_ij.argKmin(k + 1, dim=1)  # shape: (N, k+1)
    # Remove self-match and bring to CPU.
    neighbors_pykeops = neighbors_pykeops[:, 1:].cpu().long()  # shape: (N, k)

    # Compute the corresponding distances using standard PyTorch operations.
    # For each point i and neighbor j, compute: ||X[i] - X[neighbors_pykeops[i,j]]||^2.
    row_idx = torch.arange(N).unsqueeze(1).expand_as(neighbors_pykeops)
    X_neighbors = X[neighbors_pykeops]  # shape: (N, k, D)
    diffs = X.unsqueeze(1) - X_neighbors  # shape: (N, k, D)
    distances_pykeops = (diffs ** 2).sum(-1)  # shape: (N, k)

    # -------------------------------
    # 3. Build AMPI index on MNIST.
    # -------------------------------
    # Convert the data to NumPy for AMPI.
    X_np = X.numpy().astype(np.float32)
    ampi = AMPIIndex(X_np, num_projections)

    # -------------------------------
    # 4. For a subset of queries, compare AMPI and PyKeOps results.
    # -------------------------------
    # Select random query indices.
    rng = np.random.default_rng(42)
    query_indices = rng.choice(N, size=num_queries, replace=False)

    overlap_ratios = []
    distance_error_ratios = []
    ampi_runtime = []

    for count, qi in enumerate(query_indices, start=1):
        # Get query vector (both torch and numpy versions).
        q_torch = X[qi]            # for potential Torch operations
        q_np    = X_np[qi]         # for AMPI

        # PyKeOps results for this query.
        pykeops_indices = neighbors_pykeops[qi].numpy()  # shape: (k,)
        pykeops_dists   = distances_pykeops[qi].numpy()    # shape: (k,)

        # AMPI approximate search.
        start_ampi = time.perf_counter_ns()
        approx_points, approx_dists, approx_indices = ampi.query(q_np, window_size, k+1)
        end_ampi = time.perf_counter_ns()
        ampi_runtime.append(end_ampi - start_ampi)

        # Ensure AMPI results are NumPy arrays.
        approx_indices = np.array(approx_indices)[1:]
        approx_dists   = np.array(approx_dists)[1:]

        print(f"\nQuery {count}/{num_queries} (Index: {qi}):")
        print("  PyKeOps  -> distances:", pykeops_dists, ", indices:", pykeops_indices)
        if len(approx_dists) > 0:
            print("  AMPI     -> distances:", approx_dists, ", indices:", approx_indices)
        else:
            print("  AMPI returned no neighbors.")

        # Compute the index overlap ratio (fraction of common indices).
        common = set(pykeops_indices).intersection(set(approx_indices))
        overlap = len(common) / k
        overlap_ratios.append(overlap)
        print(f"  Index overlap ratio: {overlap:.2f}")

        # For distance comparison, sort both distance arrays (since the order may differ).
        sorted_pykeops = np.sort(pykeops_dists)
        sorted_ampi    = np.sort(approx_dists)
        # Compute the element-wise ratio (AMPI / PyKeOps), taking care to avoid division by zero.
        ratio = np.divide(sorted_ampi, sorted_pykeops, out=np.ones_like(sorted_ampi), where=(sorted_pykeops != 0))
        mean_ratio = np.mean(ratio)
        distance_error_ratios.append(mean_ratio)
        print(f"  Mean distance error ratio (AMPI/pykeops): {mean_ratio:.6f}")

    avg_overlap = np.mean(overlap_ratios)
    avg_distance_ratio = np.mean(distance_error_ratios)
    avg_ampi_time = np.mean(ampi_runtime)

    print("\n=== Summary ===")
    print(f"Average index overlap ratio: {avg_overlap:.6f}")
    print(f"Average distance error ratio (AMPI/pykeops): {avg_distance_ratio:.6f}")
    print(f"Average AMPI query runtime: {avg_ampi_time/num_queries:.3f} ns")

    tolerance = 1.10  # Allow up to 10% increase in distance.
    if avg_distance_ratio > tolerance:
        print(f"AMPI's average distance ratio {avg_distance_ratio:.6f} exceeds the tolerance {tolerance}")
    else:
        print(f"AMPI's average distance ratio {avg_distance_ratio:.6f} is within the tolerance {tolerance}")

if __name__ == "__main__":
    compare_ampi_pykeops_mnist(k=50, num_projections=512, window_size=128, num_queries=1000)

Streaming output truncated to the last 5000 lines.
 57751 67238 44417   583 54917 14597 44117 23911 32247 41615 12373 66609
 41551 13755 41677 41131 30181  3507 19199    79 41379  1057 54995 30022
 34959 22613  7287 43307 24263 17359 31100 44323 44361 53951 29867 31439
 44367 59746]
  AMPI     -> distances: [14.064852 14.884998 15.258686 16.722    16.766321 18.039078 19.037325
 19.108051 19.14742  19.711649 20.050797 20.159016 20.568027 20.714972
 21.15068  21.351204 21.66316  21.927858 22.227606 22.361923 22.429588
 22.61178  22.673433 22.757402 22.766798 22.832357 23.136702 23.302254
 23.335196 23.412981 23.463268 23.628681 23.63934  23.692488 23.797049
 24.004414 24.0219   24.03597  24.354725 24.561785 24.73627  24.77687
 24.828852 24.831343 24.855425 24.87148  24.927612 24.953756 25.043583
 25.065975] , indices: [41461 41549 62123 54937  8287 22279 15991  7783  7769 56077  9927 54971
 57751 67238 44417   583 54917 14597 44117 23911 32247 41615 12373 66609
 41551 13755 41677 41131 3